# 04. Indicators

This notebook builds the four indicator tables and checks that they are
internally consistent.

1. Create the database, build its tables from `sql/schema.sql` and load the CSV
   files listed in `sql/load.sql`.
2. Run `sql/indicators.sql`, which creates one table per indicator module.
3. Check the tables against each other.

## Settings


In [ ]:
import os
import getpass
import pathlib
import sys

sys.path.append("..")

DB = {
    "host": "localhost",
    "port": 5432,
    "dbname": "madrid_crime",
    "user": "postgres",
    # Read from the environment, asked for interactively only if it is absent.
    "password": os.environ.get("PGPASSWORD") or getpass.getpass("PostgreSQL password: "),
}

SCHEMA_FILE = "../sql/schema.sql"
SQL_FILE = "../sql/indicators.sql" 
DATA_DB = "../data/db/" 

In [ ]:
import warnings
import pandas as pd
import psycopg2
from src.utils import q


warnings.filterwarnings("ignore", message=".*SQLAlchemy.*")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

## Create the database and load it

### Create and connect to the database

In [ ]:
admin = psycopg2.connect(**{**DB, "dbname": "postgres"})
admin.autocommit = True

with admin.cursor() as cur:
    cur.execute("SELECT 1 FROM pg_database WHERE datname = %s", (DB["dbname"],))
    exists = cur.fetchone() is not None
    cur.execute(f'DROP DATABASE "{DB["dbname"]}" WITH (FORCE)')
    cur.execute(f'CREATE DATABASE "{DB["dbname"]}"')

admin.close()

conn = psycopg2.connect(**DB)
conn.autocommit = True
print("connected to", DB["dbname"])

### Create tables from `schema.sql`

In [ ]:
script = pathlib.Path(SCHEMA_FILE).read_text(encoding="utf-8")

with conn.cursor() as cur:
    cur.execute(script)

q("""
    SELECT table_name
    FROM   information_schema.tables
    WHERE  table_schema = 'public' AND table_type = 'BASE TABLE'
    ORDER  BY table_name
""", conn)

### Load the tables

In [ ]:
tables_order = [
    "municipality",
    "crime_type_mun",
    "crime_type_reg",
    "mun_population",
    "mun_crime",
    "reg_crime",
    "reg_offences",
    "reg_victims",
    "reg_offenders",
]

for table in tables_order:
    path = pathlib.Path(DATA_DB) / f"{table}.csv"

    with conn.cursor() as cur, path.open(encoding="utf-8", newline="") as handle:
        cur.copy_expert(f"COPY {table} FROM STDIN WITH (FORMAT csv, HEADER true)", handle)


q(" UNION ALL ".join(
    f"SELECT '{table}' AS table_name, count(*) AS rows FROM {table}"
    for table in tables_order
) + " ORDER BY 1", conn)

## Build the indicator tables

This runs `indicators.sql` , every table is dropped and
rebuilt


In [ ]:
script = pathlib.Path(SQL_FILE).read_text(encoding="utf-8")

with conn.cursor() as cur:
    cur.execute(script)

q("""
SELECT 'ind_crime_level' AS table_name, count(*) AS rows FROM ind_crime_level
UNION ALL SELECT 'ind_police_performance', count(*) FROM ind_police_performance
UNION ALL SELECT 'ind_demographic_profile', count(*) FROM ind_demographic_profile
UNION ALL SELECT 'ind_crime_structure', count(*) FROM ind_crime_structure
UNION ALL SELECT 'ind_crime_specialisation', count(*) FROM ind_crime_specialisation
ORDER BY 1
""", conn)

In [ ]:
conn.close()
print("done")

### 9 base tables + 5 indicator tables created and loaded into de database